In [1]:
import os
import h5py
import json
import glob
import actipy
import argparse
import numpy as np
import pandas as pd
from pyedflib.edfreader import EdfReader
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.interpolate import interp1d
from scipy.signal import butter, filtfilt

from stages import io_utils
from utils.preprocess_actigraphy import preprocess_actigraphy_df
from utils.preprocess_psg import preprocess_psg_dict
from utils.write_h5 import write_h5_acc

In [2]:
# For apple michigan
labels = glob.glob('/oak/stanford/groups/mignot/actigraphy/apple_michigan/files/sleep-accel/1.0.0/labels/*.txt')
labels.sort()
acc = glob.glob('/oak/stanford/groups/mignot/actigraphy/apple_michigan/files/sleep-accel/1.0.0/motion/*.txt')
acc.sort()
hr = glob.glob('/oak/stanford/groups/mignot/actigraphy/apple_michigan/files/sleep-accel/1.0.0/heart_rate/*.txt')
hr.sort()
subject_ids = [os.path.basename(path).split('_')[0] for path in acc]
subject_ids.sort()

data_paths = pd.DataFrame(
    index = subject_ids,
    data={
        'acc_path': acc,
        'labels_path': labels,
        'hr_path': hr,
    }
)

In [8]:
print('data_len_drops', data_len_drops)
print('sample_rate_drops', sample_rate_drops)
print('neg_sample_rate_drops', neg_sample_rate_drops)
print('missing_label_drops', missing_label_drops)

data_len_drops ['3509524_acceleration.txt', '5132496_acceleration.txt', '759667_acceleration.txt']
sample_rate_drops ['5383425_acceleration.txt', '8258170_acceleration.txt']
neg_sample_rate_drops ['1360686_acceleration.txt', '1449548_acceleration.txt', '1818471_acceleration.txt', '2598705_acceleration.txt', '4018081_acceleration.txt', '4314139_acceleration.txt', '8000685_acceleration.txt', '8530312_acceleration.txt', '9618981_acceleration.txt']
missing_label_drops ['2598705_acceleration.txt', '7749105_acceleration.txt', '8000685_acceleration.txt', '8530312_acceleration.txt', '9618981_acceleration.txt']


In [9]:
stages_count

{'wake': np.float64(12.666),
 'n1': np.float64(9.971),
 'n2': np.float64(75.345),
 'n3': np.float64(18.969),
 'rem': np.float64(30.247000000000003),
 'missing': np.float64(1.158)}

In [6]:
# Options
keep_threshold = 60*60*4
fs_out = 30
gap_threshold_sec = 5
join_threshold = 2*60*60 # 10 min
out_dir = '/oak/stanford/groups/mignot/projects/actigraphy_fm/data/sleepaccel/'
chunk_size_sec = 600

data_len_drops = []
sample_rate_drops = []
neg_sample_rate_drops = [] 
missing_label_drops = []
stages_count = {
    'wake': 0,
    'n1': 0,
    'n2': 0,
    'n3': 0,
    'rem': 0,
    'missing': 0
}
for i, file in enumerate(acc):
#     if not os.path.basename(file).split('_')[0] in ['759667', '7749105']:
#         continue
    acc_df = pd.read_csv(file, sep = ' ', header = None).rename(
        columns = {
        0: 'sec_from_start',
        1: 'x',
        2: 'y',
        3: 'z',
    })

    psg_start = acc_df['sec_from_start'].abs().argmin()
    acc_df = acc_df.iloc[psg_start-1:]
    acc_df = acc_df.set_index(keys=['sec_from_start'])
    
    # Find sampling gaps before resampling and interpolation
    gap_end = pd.Series(acc_df.index.diff() > gap_threshold_sec)
    gap_start = gap_end.shift(-1, fill_value=False)
    gap_end = np.argwhere(gap_end.values)
    gap_start = np.argwhere(gap_start.values)
    gap_end_idx = list(acc_df.iloc[gap_end.ravel()].index)
    gap_start_idx = list(acc_df.iloc[gap_start.ravel()].index)


    # Get median sample frequency
    sample_gap = acc_df.index.diff()[1:]
    median_gap = np.median(sample_gap)
    q5, q95 = np.percentile(sample_gap, [5, 95])
    median_fs = 1/median_gap
    print('Median sampling freq:', median_fs)
    print('q5 and q95 sampling freq', 1/q5, 1/q95,'\n')

    if median_fs < fs_out:
        print(f'skipping {os.path.basename(file)} due to low sampling frequency\n')
        sample_rate_drops.append(os.path.basename(file))
        continue

    if (acc_df.index.diff() < 0).sum() > 0:
        print(f'skipping {os.path.basename(file)} due to non-increasing timestamps\n')
        neg_sample_rate_drops.append(os.path.basename(file))
   
    # Interpolate to 50 Hz uniform sampling
    interp = interp1d(acc_df.index, acc_df[['x','y','z']], axis=0)
    idx_uniform = np.linspace(acc_df.index[0], acc_df.index[-1], int((acc_df.index[-1]-acc_df.index[0])//median_gap))#int(new_start)+1, int(new_end), median_gap)
    signal_uniform = interp(idx_uniform)

    # Apply anti-aliasing filter
    filter_order = 4
    b, a = butter(filter_order, fs_out/median_fs, btype='lowpass')
    filtered_signal = filtfilt(b, a, signal_uniform, axis=0)

    # Resample to 30 Hz
    interp = interp1d(idx_uniform, filtered_signal, axis=0)
    num_samples = int(np.round(idx_uniform[-1] * fs_out))
    new_time_points = np.linspace(idx_uniform[0], idx_uniform[-1], num = num_samples)
    resampled_signal = interp(new_time_points)

    acc_df_new = pd.DataFrame(index=new_time_points, data={
        'x': resampled_signal[:,0],
        'y': resampled_signal[:,1],
        'z': resampled_signal[:,2],
    })

    # Read labels
    print('Acc and label match', os.path.basename(file).split('_')[0] == os.path.basename(labels[i]).split('_')[0])
    label_df = pd.read_csv(labels[i], sep = ' ', header = None)
    label_df = label_df.rename(columns = {0: 'sec_from_start', 1: 'hypnogram'})

    label_df = label_df.set_index(keys=['sec_from_start'])
    label_df.index = label_df.index.astype(float)

    # Merge to resample hypnogram to sampling rate
    merge_df = pd.merge_asof(
        acc_df_new,
        label_df,
        left_index = True,
        right_index = True,
        tolerance = 31,
    )

    # Remove sampling gaps
    i = 0
    new_gaps = []
    # gap_start_idx.insert(0, float(merge_df.index[0]))
    # gap_end_idx.insert(0, float(merge_df.index[10]))
    # gap_start_idx.append(float(merge_df.index[-10]))
    # gap_end_idx.append(float(merge_df.index[-1]))

    # first = merge_df.index[0]
    # last = merge_df.index[-1]
    while i < len(gap_start):
        start = gap_start_idx[i]
        end = gap_end_idx[i]
        if start - merge_df.index[0] < join_threshold:
            start = merge_df.index[0]

        if i+1 < len(gap_start):
            next_gap_start = gap_start_idx[i+1]
            next_gap_end = gap_end_idx[i+1]
            # print("Gap", start, end)
            while (next_gap_end - end) < join_threshold:
                end = next_gap_end
                i += 1
                if i == len(gap_start)-1:
                    break
                next_gap_start = gap_start_idx[i+1]
                next_gap_end = gap_end_idx[i+1]
                # print("Incremented gap:", start, end)

        if merge_df.index[-1] - end < join_threshold:
            end = merge_df.index[-1]

        merge_df.loc[start:end] = np.nan
        new_gaps.append((start,end))
        i += 1

    # Fill NANs caused by inference tolerance at the end of signal
    merge_df['hypnogram'] = merge_df['hypnogram'].fillna(-1)
    missing = merge_df['hypnogram'] == -1

    # Trim segments with all missing labels
    last_non_missing = merge_df.index[~missing].max()
    first_non_missing = merge_df.index[~missing].min()
    merge_df = merge_df.loc[first_non_missing:last_non_missing]

    missing_frac = missing.sum()/len(missing)
    if missing_frac > 0.5:
        print(f'skipping {os.path.basename(file)} due to {missing_frac*100}% missing labels\n')
        missing_label_drops.append(os.path.basename(file))
        continue

    if len(merge_df)/fs_out < keep_threshold:
        print(f'skipping {os.path.basename(file)} due to only {len(merge_df)/fs_out/60**2} hours of data remaining\n')
        data_len_drops.append(os.path.basename(file))
        continue

    
    # fig, axes = plt.subplots(3, 1, figsize = (10,6), layout='constrained')
    # hour_index = merge_df.index/60/60
    # axes[0].set_yticks(ticks = [-1,0,1,2,3,4,5], labels=['missing', 'wake', 'n1', 'n2', 'n3', 'n4', 'rem'])
    # axes[0].plot(hour_index, merge_df['hypnogram'])
    # axes[0].grid()
    # axes[1].plot(hour_index, np.square(merge_df[['x', 'y', 'z']].values).sum(axis=1))
    # axes[1].set_ylim(-0.2, 3)
    # axes[2].plot(hour_index, merge_df[['x', 'y', 'z']])
    # axes[2].set_xlabel('hours')
    # # for i in range(len(gap_start_idx)):
    # #     for ax in axes:
    # #         ax.axvline(x=gap_start_idx[i]/60**2, color='r', lw=1)
    # #         ax.axvline(x=gap_end_idx[i]/60**2, color='b', lw=1)
    # fig.suptitle(f'{os.path.basename(file).split('_')[0]} preprocessed')
    # plt.show()


    # Make grouped annotations
    annotations = merge_df[['hypnogram']]

    annotations.loc[:,'hypnogram'] = annotations['hypnogram'].map({
        -1: 'missing',
        0: 'wake',
        1: 'n1',
        2: 'n2',
        3: 'n3',
        4: 'n3',
        5: 'rem',
    })
    annotations.loc[:,'sleep_wake'] = annotations['hypnogram'].map({
        'wake':'wake',
        'n1':'sleep',
        'n2':'sleep',
        'n3':'sleep',
        'rem':'sleep',
        'missing':'missing',
    })
    annotations.loc[:,'rem_nrem'] = annotations['hypnogram'].map({
        'wake':'wake',
        'n1':'nrem',
        'n2':'nrem',
        'n3':'nrem',
        'rem':'rem',
        'missing':'missing',
    })
    annotations.loc[:,'light_deep_rem'] = annotations['hypnogram'].map({
        'wake':'wake',
        'n1':'light',
        'n2':'light',
        'n3':'deep',
        'rem':'rem',
        'missing':'missing',
    })

    # Get sleep stage amounts
    stages_hours = {}
    for stage in annotations['hypnogram'].unique():
        stage_hours = round(
            (annotations['hypnogram'] == stage).sum() 
            / fs_out / 60**2,
            3
        )
        stages_count[stage] += stage_hours
        stages_hours[stage] = stage_hours

    outfile = f'{out_dir}/{os.path.basename(file).split('_')[0]}.h5'
    write_h5_acc(
        outfile,
        accelerometry = merge_df[['x', 'y', 'z']],
        x_col = 'x',
        y_col = 'y',
        z_col = 'z',
        acc_info = {},
        annotations = annotations,
        study_start = merge_df.index[0],
        chunk_size_sec = 600,
        ahi = None,
        stage_hours = stages_hours,
    )
    

Median sampling freq: 49.985154409342705
q5 and q95 sampling freq 50.74839160585179 49.19081115880502 

Acc and label match True


/tmp/ipykernel_18836/2217519854.py:178: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['wake' 'wake' 'wake' ... 'wake' 'wake' 'wake']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  annotations.loc[:,'hypnogram'] = annotations['hypnogram'].map({
/tmp/ipykernel_18836/2217519854.py:187: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations.loc[:,'sleep_wake'] = annotations['hypnogram'].map({
/tmp/ipykernel_18836/2217519854.py:195: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.p

Median sampling freq: 49.9797831790421
q5 and q95 sampling freq 50.76863716349103 49.168898114952675 

skipping 1360686_acceleration.txt due to non-increasing timestamps

Acc and label match True


/tmp/ipykernel_18836/2217519854.py:178: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['rem' 'rem' 'rem' ... 'n2' 'n2' 'n2']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  annotations.loc[:,'hypnogram'] = annotations['hypnogram'].map({
/tmp/ipykernel_18836/2217519854.py:187: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations.loc[:,'sleep_wake'] = annotations['hypnogram'].map({
/tmp/ipykernel_18836/2217519854.py:195: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org

Median sampling freq: 50.35246727926985
q5 and q95 sampling freq 100.97950116001903 49.563838223651196 

skipping 1449548_acceleration.txt due to non-increasing timestamps

Acc and label match True


...

/tmp/ipykernel_18836/2217519854.py:178: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['wake' 'wake' 'wake' ... 'rem' 'rem' 'rem']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  annotations.loc[:,'hypnogram'] = annotations['hypnogram'].map({
/tmp/ipykernel_18836/2217519854.py:187: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations.loc[:,'sleep_wake'] = annotations['hypnogram'].map({
/tmp/ipykernel_18836/2217519854.py:195: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pyda

Median sampling freq: 49.97800967993273
q5 and q95 sampling freq 50.653377921314906 49.27054951113679 

Acc and label match True


/tmp/ipykernel_18836/2217519854.py:178: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['n3' 'n3' 'n3' ... 'n2' 'n2' 'n2']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  annotations.loc[:,'hypnogram'] = annotations['hypnogram'].map({
/tmp/ipykernel_18836/2217519854.py:187: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations.loc[:,'sleep_wake'] = annotations['hypnogram'].map({
/tmp/ipykernel_18836/2217519854.py:195: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pa

Median sampling freq: 33.410288363996514
q5 and q95 sampling freq 67.1812752128354 33.11806590564576 

skipping 9618981_acceleration.txt due to non-increasing timestamps

Acc and label match True
skipping 9618981_acceleration.txt due to 88.5706902557068% missing labels

Median sampling freq: 51.00739607372836
q5 and q95 sampling freq 94.35796013165104 49.65711760141734 

Acc and label match True


/tmp/ipykernel_18836/2217519854.py:178: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['wake' 'wake' 'wake' ... 'wake' 'wake' 'wake']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  annotations.loc[:,'hypnogram'] = annotations['hypnogram'].map({
/tmp/ipykernel_18836/2217519854.py:187: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations.loc[:,'sleep_wake'] = annotations['hypnogram'].map({
/tmp/ipykernel_18836/2217519854.py:195: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.p

In [ ]:
# Options
keep_threshold = 60*60*4
fs_out = 30
gap_threshold_sec = 5
# out_dir = '/oak/stanford/groups/mignot/projects/actigraphy_fm/sleepaccel_data/'
chunk_size_sec = 600

data_len_drops = []
sample_rate_drops = []
neg_sample_rate_drops = [] 
missing_label_drops = []

for i, file in enumerate(acc):
    acc_df = pd.read_csv(file, sep = ' ', header = None).rename(
        columns = {
        0: 'sec_from_start',
        1: 'x',
        2: 'y',
        3: 'z',
    })
    # start_time = pd.Timestamp(year=2024, month=6, day=10, hour=22)

    psg_start = acc_df['sec_from_start'].abs().argmin()
    acc_df = acc_df.iloc[psg_start-1:]
    acc_df = acc_df.set_index(keys=['sec_from_start'])

    # Find longest contiguous recording segment
    gap_end = pd.Series(acc_df.index.diff() > gap_threshold_sec)
    gap_start = gap_end.shift(-1, fill_value=False)
    gap_end = np.argwhere(gap_end.values)
    gap_start = np.argwhere(gap_start.values)
    gap_end_idx = acc_df.iloc[gap_end.ravel()].index
    gap_start_idx = acc_df.iloc[gap_start.ravel()].index


    # if gap_end.sum() == 0:
    #     new_start = acc_df.index[0]
    #     new_end = acc_df.index[-1]
    
    # elif gap_end.sum() >= 1:
    #     gap_end_idx = acc_df.index[gap_end]
    #     gap_start_idx = acc_df.index[gap_start]

    #     first_segment = (gap_start_idx[0] - acc_df.index[0])
    #     last_segment = (acc_df.index[-1] - gap_end_idx[-1])
    #     longest_time = 0
    #     if gap_end.sum() > 1:
    #         contiguous_segments = (gap_start_idx[1:] - gap_end_idx[:-1])
    #         longest_time = contiguous_segments.max()
    #         longest_idx = contiguous_segments.argmax()

    #     if longest_time > first_segment and longest_time > last_segment:
    #         new_start = gap_end_idx[:-1][longest_idx]
    #         new_end = gap_start_idx[1:][longest_idx]
    #     elif first_segment > last_segment:
    #         new_start = acc_df.index[0]
    #         new_end = gap_start_idx[0]
    #     else:
    #         new_start = gap_end_idx[-1]
    #         new_end = acc_df.index[-1]

    # Read labels
    print('Acc and label match', os.path.basename(file).split('_')[0] == os.path.basename(labels[i]).split('_')[0])
    label_df = pd.read_csv(labels[i], sep = ' ', header = None)
    label_df = label_df.rename(columns = {0: 'sec_from_start', 1: 'hypnogram'})

    label_df = label_df.set_index(keys=['sec_from_start'])
    label_df.index = label_df.index.astype(float)

    # # Plot data and labels
    # hour_index_acc = acc_df.index/60/60
    # hour_index_label = label_df.index/60/60
    # fig, axes = plt.subplots(3, 1, figsize = (10,6), layout='constrained',sharex=True)
    # axes[0].plot(hour_index_label, label_df['hypnogram'])
    # axes[0].set_yticks(ticks = [-1,0,1,2,3,4,5], labels=['missing', 'wake', 'n1', 'n2', 'n3', 'n4', 'rem'])
    # axes[0].grid()
    # axes[1].plot(hour_index_acc, np.square(acc_df[['x', 'y', 'z']].values).sum(axis=1))
    # axes[1].set_ylim(-0.2, 3)
    # axes[1].vlines(hour_index_acc[gap_start], ymin=0, ymax=2, color='b', ls='--')
    # axes[1].vlines([new_start/60/60, new_end/60/60], ymin=0, ymax=2, color='r')
    # axes[2].plot(hour_index_acc, acc_df[['x', 'y', 'z']])
    # axes[2].set_xlabel('hours')
    # fig.suptitle(os.path.basename(file).split('_')[0])
    # plt.show()

    # if (new_end - new_start) < keep_threshold:
    #     print(f'skipping {os.path.basename(file)} due to isufficient data\n')
    #     data_len_drops.append(os.path.basename(file))
    #     continue
    

    # # Cut out new segment
    # acc_df_new = acc_df.loc[new_start : new_end]
    # label_df_new = label_df.loc[new_start-30 : new_end+30]

    # sample_gap = acc_df_new.index.diff()[1:]
    # median_gap = np.median(sample_gap)
    # q5, q95 = np.percentile(sample_gap, [5, 95])
    # median_fs = 1/median_gap
    # print('Median sampling freq:', median_fs)
    # print('q5 and q95 sampling freq', 1/q5, 1/q95,'\n')

    # if median_fs < fs_out:
    #     print(f'skipping {os.path.basename(file)} due to low sampling frequency\n')
    #     sample_rate_drops.append(os.path.basename(file))
    #     continue

    # if (acc_df_new.index.diff() < 0).sum() > 0:
    #     print(f'skipping {os.path.basename(file)} due to non-increasing timestamps\n')
    #     neg_sample_rate_drops.append(os.path.basename(file))
    #     continue

    # Interpolate to 50 Hz uniform sampling
    interp = interp1d(acc_df_new.index, acc_df_new[['x','y','z']], axis=0)
    num_samples = acc_df_new.index[-1]
    idx_uniform = np.arange(int(new_start)+1, int(new_end), median_gap)
    signal_uniform = interp(idx_uniform)

    # Apply anti-aliasing filter
    filter_order = 4
    b, a = butter(filter_order, fs_out/median_fs, btype='lowpass')
    filtered_signal = filtfilt(b, a, signal_uniform, axis=0)

    # Resample to 30 Hz
    interp = interp1d(idx_uniform, filtered_signal, axis=0)
    num_samples = int(np.round(idx_uniform[-1] * fs_out))
    new_time_points = np.linspace(idx_uniform[0], idx_uniform[-1], num = num_samples)
    resampled_signal = interp(new_time_points)

    acc_df_new = pd.DataFrame(index=new_time_points, data={
        'x': resampled_signal[:,0],
        'y': resampled_signal[:,1],
        'z': resampled_signal[:,2],
    })

    # Merge to resample hypnogram to sampling rate
    merge_df = pd.merge_asof(
        acc_df_new,
        label_df_new,
        left_index = True,
        right_index = True,
        tolerance = 31,
    )
    # Fill NANs cause by inference tolerance at the end of signal
    merge_df['hypnogram'] = merge_df['hypnogram'].fillna(-1)
    missing = merge_df['hypnogram'] == -1

    missing_frac = missing.sum()/len(missing)
    if missing_frac > 0.5:
        print(f'skipping {os.path.basename(file)} due to {missing_frac*100}% missing labels\n')
        missing_label_drops.append(os.path.basename(file))
        continue
    
    last_non_missing = merge_df.index[~missing].max()
    merge_df = merge_df.loc[:last_non_missing]

    fig, axes = plt.subplots(3, 1, figsize = (10,6), layout='constrained')
    hour_index = merge_df.index/60/60
    axes[0].set_yticks(ticks = [-1,0,1,2,3,4,5], labels=['missing', 'wake', 'n1', 'n2', 'n3', 'n4', 'rem'])
    axes[0].plot(hour_index, merge_df['hypnogram'])
    axes[0].grid()
    axes[1].plot(hour_index, np.square(merge_df[['x', 'y', 'z']].values).sum(axis=1))
    axes[1].set_ylim(-0.2, 3)
    axes[2].plot(hour_index, merge_df[['x', 'y', 'z']])
    axes[2].set_xlabel('hours')
    fig.suptitle(f'{os.path.basename(file).split('_')[0]} preprocessed')
    plt.show()

    # Make grouped annotations
    annotations = merge_df[['hypnogram']]

    annotations.loc[:,'hypnogram'] = annotations['hypnogram'].map({
        -1: 'missing',
        0: 'wake',
        1: 'n1',
        2: 'n2',
        3: 'n3',
        4: 'n3',
        5: 'rem',
    })
    annotations.loc[:,'sleep_wake'] = annotations['hypnogram'].map({
        'wake':'wake',
        'n1':'sleep',
        'n2':'sleep',
        'n3':'sleep',
        'rem':'sleep',
        'missing':'missing',
    })
    annotations.loc[:,'rem_nrem'] = annotations['hypnogram'].map({
        'wake':'wake',
        'n1':'nrem',
        'n2':'nrem',
        'n3':'nrem',
        'rem':'rem',
        'missing':'missing',
    })
    annotations.loc[:,'light_deep_rem'] = annotations['hypnogram'].map({
        'wake':'wake',
        'n1':'light',
        'n2':'light',
        'n3':'deep',
        'rem':'rem',
        'missing':'missing',
    })

    # out_file = f'{out_dir}/{os.path.basename(file).split('_')[0]}.h5'
    # with h5py.File(out_file, 'w') as f:
    #     f.create_group('annotations')
    #     f.create_group('data')
    #     f.attrs.create('start_time', merge_df.index[0])

    #     # Write the calibrated accelerometry data
    #     acc_fs = round(1/merge_df.index.diff()[1:].to_numpy().mean(), 2)
    #     acc_val = merge_df[['x', 'y', 'z']].values
    #     dataset = f['data'].create_dataset(
    #         f'accelerometry',
    #         data=acc_val.T.astype(np.float32),
    #         chunks=(acc_val.shape[1], int(chunk_size_sec * acc_fs)),
    #     )
    #     dataset.attrs.create('sample_frequency', acc_fs)

    #     # Write the annotations
    #     for field in annotations.columns:
    #         annotation = annotations[field]
    #         dataset = f['annotations'].create_dataset(
    #             f'{field}',
    #             data=annotation.to_numpy(),
    #             dtype=h5py.string_dtype(encoding='utf-8'),
    #             chunks=(chunk_size_sec * acc_fs,),
    #         )
    #         dataset.attrs.create('sample_frequency', acc_fs)

    